# Kernel C-restB
Runs ONLY the 2 class-balanced configs that did not finish earlier (powers=(1.0,)).
max_repeat=12 to match the completed p=1.0 run so results stay comparable.


In [ ]:
import os, sys, tarfile, glob, time, json, subprocess
from pathlib import Path
T0=time.time()
def clock(msg): print(f'[{time.time()-T0:7.1f}s] {msg}', flush=True)

LABSD_DIR='/kaggle/working/labsd_pkg'; Path(LABSD_DIR).mkdir(parents=True,exist_ok=True)
def locate():
    for init in glob.glob('/kaggle/input/**/labsd/__init__.py',recursive=True):
        return str(Path(init).parent.parent)
    for tar in glob.glob('/kaggle/input/**/labsd.tar',recursive=True):
        with tarfile.open(tar) as tf: tf.extractall(LABSD_DIR)
        for init in glob.glob(LABSD_DIR+'/**/labsd/__init__.py',recursive=True):
            return str(Path(init).parent.parent)
    return None
P=locate(); assert P, 'labsd not found'
if P not in sys.path: sys.path.insert(0,P)
import labsd; from labsd import splits as _s, class_balance as _cb
clock('labsd loaded '+labsd.__file__)


In [ ]:
# install nuscenes-devkit (no-deps to protect kaggle numpy/scipy) + ultralytics
clock('installing deps...')
subprocess.run([sys.executable,'-m','pip','install','--no-deps','--no-cache-dir','nuscenes-devkit'],capture_output=True,text=True)
subprocess.run([sys.executable,'-m','pip','install','--no-cache-dir','pyquaternion','cachetools','descartes','shapely'],capture_output=True,text=True)
r=subprocess.run([sys.executable,'-m','pip','install','--no-deps','--no-cache-dir','ultralytics'],capture_output=True,text=True)
subprocess.run([sys.executable,'-m','pip','install','--no-cache-dir','ultralytics-thop','py-cpuinfo'],capture_output=True,text=True)
clock('deps installed')


In [ ]:
# locate full nuScenes trainval mount
meta=None
for dp,dn,fn in os.walk('/kaggle/input'):
    if 'scene.json' in fn and 'log.json' in fn and 'trainval' in dp: meta=dp; break
assert meta, 'trainval metadata not found'
NUSC_ROOT=str(Path(meta).parent)   # dir that CONTAINS v1.0-trainval and samples/
clock('nuScenes root: '+NUSC_ROOT)
from nuscenes.nuscenes import NuScenes
clock('loading NuScenes devkit (this is slow on trainval)...')
NUSC=NuScenes(version='v1.0-trainval', dataroot=NUSC_ROOT, verbose=False)
clock(f'NuScenes loaded: {len(NUSC.scene)} scenes')


In [ ]:
# build full splits, then cap val to 50 scenes
from labsd.splits import partition_by_location, cap_val_scenes, count_val_frames
logs_by={l['token']:l for l in NUSC.log}
full=partition_by_location(NUSC.scene, logs_by, train_ratio=0.6)
clock(f"full splits: bos_tr={len(full['boston_train'])} bos_val={len(full['boston_val'])} "
      f"sg_tr={len(full['singapore_train'])} sg_val={len(full['singapore_val'])}")
capped=cap_val_scenes(full, NUSC, max_val_scenes=50)
nfr=count_val_frames(capped, NUSC, 'singapore_val')
clock(f"capped sg_val = {len(capped['singapore_val'])} scenes = {nfr} frames (was 3 in mini)")
SPLITS_JSON='/kaggle/working/splits_full.json'
json.dump(capped, open(SPLITS_JSON,'w'))
clock('wrote '+SPLITS_JSON)


In [ ]:
# train Boston incumbent C1 on FULL boston_train, then one baseline measurement
from labsd.c1_yolo import build_yolo_dataset, write_yolo_data_yaml, fine_tune_yolo, c1_descriptor_yolo
from labsd.train_c2 import train_c2
from labsd.eval import run_all_measurements

DS='/kaggle/working/boston_ds'
clock('materialising boston_train images (full)...')
nb_img=build_yolo_dataset(NUSC, capped['boston_train'], DS, split_name='train')
# small val for YOLO's own mAP: reuse singapore_val scenes
build_yolo_dataset(NUSC, capped['singapore_val'], DS, split_name='val')
yaml=write_yolo_data_yaml(DS)
clock(f'boston train images: {nb_img}')
t=time.time()
best=fine_tune_yolo(data_yaml=yaml, out_dir='/kaggle/working/c1_boston', base_weights='yolo11n.pt',
                    epochs=10, name='c1_boston_full', seed=0)
clock(f'Boston C1 trained in {time.time()-t:.0f}s -> {best}')
c1_boston=c1_descriptor_yolo('/kaggle/working/c1_boston', best, label='boston_full', val_data_yaml=yaml)
c2_boston=train_c2(split='boston_train', out_path='/kaggle/working/c2_boston.json')
t=time.time()
clock('running baseline measurement on 50-scene singapore_val (times the eval cost)...')
base=run_all_measurements(c1_ckpt=c1_boston, c2_ckpt=c2_boston, split='singapore_val',
                          out_path='/kaggle/working/baseline_full.json',
                          nusc=NUSC, splits_json=SPLITS_JSON, enable_pkl=False)
clock(f'baseline eval done in {time.time()-t:.0f}s')
print(json.dumps(base, indent=2))
json.dump({'n_val_frames':nfr,'n_val_scenes':len(capped['singapore_val']),
           'boston_train_images':nb_img,'baseline':base}, open('/kaggle/working/kernelA_summary.json','w'))
clock('=== KERNEL A DONE — see timings above to scope B and C ===')


In [ ]:
# Kernel C-restB: 2 class-balanced fine-tunes (powers=(1.0,))
from labsd.c1_yolo import build_yolo_dataset as _byd
from labsd.campaign import run_class_balanced

SG_FULL_DIR='/kaggle/working/sg_full_ds'
clock('materialising FULL singapore_train...')
n_sg=_byd(NUSC, capped['singapore_train'], SG_FULL_DIR, split_name='train')
_byd(NUSC, capped['singapore_val'], SG_FULL_DIR, split_name='val')
clock(f'singapore_train images: {n_sg}')

t=time.time()
clock('running powers=(1.0,) x epochs(20,) x seeds(2,3) = 2 runs...')
cb=run_class_balanced(nusc=NUSC, splits=capped, splits_json=SPLITS_JSON,
                      c1_boston_descriptor=c1_boston, c2_ckpt=c2_boston,
                      baseline_json='/kaggle/working/baseline_full.json',
                      sg_full_dataset_dir=SG_FULL_DIR, work_root='/kaggle/working/balanced',
                      epochs_list=(20,), seeds=(2,3), powers=(1.0,),
                      max_repeat=12, nusc_maps=None, enable_pkl=False)
clock(f'DONE in {time.time()-t:.0f}s')
json.dump(cb, open('/kaggle/working/class_balanced_restB_p10_ep20.json','w'), indent=2)
print('delta1>0 in', cb['n_delta1_positive'], '/', len(cb['rows']))
print('STRICT EE in', cb['n_strict_entangled_enhancement'], '/', len(cb['rows']))
for r in cb['rows']:
    print(f"  {r['tag']:<22} d1={(r.get('delta1') or 0):+.4f} "
          f"D3={(r.get('Delta3') or 0):+.3f} EE={r['strict_entangled_enhancement']}")
clock('=== Kernel C-restB DONE ===')

